<img title="Instituto Federal do Amazonas" alt="logo do IFAM" width="200" src='https://imgs.search.brave.com/7S6EabbIm2owts-ML11oCagljcO_OnAQik1wNzBZmU8/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly93d3cu/Z292LmJyL2lucGEv/cHQtYnIvaW5vdmFj/YW8vaW1hZ2Vucy9s/b2dvcy1leHRlcm5h/cy9pZmFtLnBuZy9A/QGltYWdlcy9pbWFn/ZS5wbmc'>

# Instituto Federal de Educação, Ciência e Tecnologia do Amazonas
## Bacharelado em Ciências da Computação
### Programação para análise de dados
#### Gabriele Silva
#### Victoria Maciel


---

1.  Definição e Validação dos Schemas

2. Operações em Grandes Volumes de dados, incrementado gradativamente
 
2.1 - Horas

2.2 - Dias

2.3 - Semanas

2.4 - Meses

2.5 - Trimestres

2.6 - Semestres

2.7 - Anos

2.8 - Limite Máximo do Dataset

> * Operações de Consultas, com agrupamentos e Geração de Gráficos 

3. Comparação de Desempenho, envolvendo as bibliotecas utilizadas em sala

3.1 - Pandas

3.2 - PyArrow

3.3 - Polars

3.4 - Dask


4. Processamento paralelo com Dask, utilizando pelo menos 3 máquinas (Físicas) diferentes e geração de relatórios de desempenho com variações de numero de workers. Também fazer com aumento gradual, como por exemplo: 1
1 máquina

2 máquinas 

3 máquinas

**{ as três variar de 2 a N workers }**

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as py

import requests
import zipfile
import io

import os, warnings
os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'
warnings.filterwarnings('ignore')

n_amostra = 500_000
fim_anos = [15, 16, 17, 18, 19, 20, 21, 22, 23, 24]

for i in fim_anos:
    
    url = f'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINASC/csv/SINASC_20{i}_csv.zip'
    
    arquivo_local = None
    url_natalidade = url
    
    fonte = arquivo_local if (arquivo_local and Path(arquivo_local).exists()) else url_natalidade
    print(f'Fonte configurada: {fonte[:80]}...' if len(fonte)>80 else f'Fonte: {fonte}')
    
    
    # Download do arquivo bruto - conteúdo binário
    resposta_da_requisicao = requests.get(url_natalidade)
    conteudo_do_arquivo = io.BytesIO(resposta_da_requisicao.content)
    
    # "Des"compactação e leitura
    with zipfile.ZipFile(conteudo_do_arquivo) as zippado:
        arquivos_internos = zippado.namelist()
        with zippado.open(arquivos_internos[0]) as f:
            df_natalidade = pd.read_csv(f, sep=';', nrows=n_amostra, encoding='latin1', low_memory=False)
            
    # Validação da Compressão
    compressao = "ZIP" if fonte.endswith('.zip') else "Nenhuma"
    
    # Verificação dos metadados
    uso_memoria = df_natalidade.memory_usage(deep=True) / 1024
    print(f'{"Coluna":<21}|{"Tipo":<12}|{"RAM":<5}')
    print(f'{"_"*50}')
    for meta in df_natalidade.columns:
        tipo = str(df_natalidade[meta].dtype)
        espaco = uso_memoria[meta]
        print(f' {meta:<20}| {tipo:<10} |{espaco:>12.1f}')
    print(f'{"_"*50}')
    print('')
    
    
    # Separação das principais colunas para análise
    lista_colunas_principais = [
        #PAIS
        'IDADEMAE',
        'IDADEPAI',
    
        #GESTAÇÃO
        'SEMAGESTAC', #SEMANAS DE GESTAÇÃO
        'QTDPARTNOR', #QUANTIDADE DE PARTOS NORMAIS
        'QTDPARTCES', #QUANTIDADE DE PARTOS CESÁRIOS
        'CONSPRENAT', #QUANTIDADE DE PRENATAIS FEITOS.
        'KOTELCHUCK', #INDICE QUE CLASSIFICA A QUALIDADE DO PRENATAL FEITO
        'DIFDATA', #Diferença de dias entre o nascimento e o registro.
        'GRAVIDEZ', #INDICA SE A GRAVIDEZ FOI SIMPLES, DUPLA, TRIPLA E ETC.
    
        #BEBÊ E NASCIMENTO
        'PESO',
        'HORANASC',
        'DTNASC',
        'SEXO'
    ]
    
    df_natalidade.columns = df_natalidade.columns.str.upper()
    
    # Verificação dos metadados
    uso_memoria = df_natalidade.memory_usage(deep=True) / 1024
    print(f'{"Coluna":<21}|{"Tipo":<12}|{"RAM":<5}')
    print(f'{"_"*50}')
    for meta in df_natalidade.columns:
        tipo = str(df_natalidade[meta].dtype)
        espaco = uso_memoria[meta]
        print(f' {meta:<20}| {tipo:<10} |{espaco:>12.1f}')
    print(f'{"_"*50}')
    print('')
    
    # Organizando um novo dataframe com as colunas
    df_final = df_natalidade[lista_colunas_principais].copy()
    
    #transformando em .parquet
    df_final.to_parquet(f'sinasc20{i}_reduzidissima_final.parquet', index=False)
    print(f'Arquivo sinasc20{i}_reduzidissima_final.parquet salvo com sucesso!!!')
    print(f'{"*"*120}\n')

tabelas = []
path = '/home/victoria/Downloads/PAD/'

for i in fim_anos:
    arquivo = pq.read_table(f"{path}sinasc20{i}_reduzidissima_final.parquet")
    tabelas.append(arquivo)

df_finalissimo = py.concat_tables(tabelas, promote_options="permissive")
pq.write_table(df_finalissimo, 'sinasc_reduzidissima_final.parquet')

print(f'Arquivo sinasc_reduzidissima_final.parquet salvo com sucesso!!!')
print(f'{"*"*120}\n')

Fonte configurada: https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINASC/csv/SINASC_2015_csv....
Coluna               |Tipo        |RAM  
__________________________________________________
 CONTADOR            | int64      |      3906.2
 ORIGEM              | int64      |      3906.2
 CODESTAB            | float64    |      3906.2
 CODMUNNASC          | int64      |      3906.2
 LOCNASC             | int64      |      3906.2
 IDADEMAE            | float64    |      3906.2
 ESTCIVMAE           | float64    |      3906.2
 ESCMAE              | float64    |      3906.2
 CODOCUPMAE          | float64    |      3906.2
 QTDFILVIVO          | float64    |      3906.2
 QTDFILMORT          | float64    |      3906.2
 CODMUNRES           | int64      |      3906.2
 GESTACAO            | float64    |      3906.2
 GRAVIDEZ            | float64    |      3906.2
 PARTO               | float64    |      3906.2
 CONSULTAS           | float64    |      3906.2
 DTNASC              | int64      |  

In [2]:
df_finalissimo.to_pandas()

,IDADEMAE,IDADEPAI,SEMAGESTAC,QTDPARTNOR,QTDPARTCES,CONSPRENAT,KOTELCHUCK,DIFDATA,GRAVIDEZ,PESO,HORANASC,DTNASC,SEXO
0,31.0,NaN,NaN,NaN,NaN,NaN,9,50,1.0,2900.0,1730.0,8012015,2
1,28.0,NaN,NaN,NaN,NaN,NaN,9,147,1.0,4150.0,815.0,3022015,2
2,26.0,NaN,NaN,NaN,NaN,NaN,9,138,1.0,3490.0,800.0,12022015,2
3,21.0,NaN,NaN,NaN,NaN,NaN,9,29,1.0,3200.0,835.0,31032015,1
4,18.0,NaN,NaN,NaN,NaN,NaN,9,15,1.0,3200.0,2119.0,14042015,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4999995,21.0,NaN,40.0,0.0,0.0,13.0,5,31,1.0,3235.0,1137.0,29062024,1
4999996,36.0,NaN,39.0,0.0,1.0,14.0,5,31,1.0,3905.0,1451.0,29062024,1
4999997,31.0,NaN,31.0,0.0,1.0,8.0,5,30,1.0,1820.0,1803.0,30062024,2
4999998,26.0,NaN,41.0,1.0,0.0,9.0,5,30,1.0,3750.0,1521.0,30062024,1


---

# 1. Definição e Validação dos Schemas

In [3]:
import pandera as pa
import time 
import polars as pl
import psutil
import pyarrow.compute as pc

In [4]:
tabela_de_comparacao = []

In [5]:
schema = pa.DataFrameSchema({
    'IDADEMAE': pa.Column(
        float, #dtype
        description = 'Idade da mãe do Recém Nascido.',
        nullable=True, # nullable=False: Proíbe valores vazios (NaN)
        checks=[pa.Check.gt(0), pa.Check.lt(100)],
        coerce=True
    ),
    'IDADEPAI': pa.Column(
        float, #dtype
        description = 'Idade da pai do Recém Nascido.',
        nullable = True, # nullable=False: Proíbe valores vazios (NaN)
        checks=[pa.Check.gt(0), pa.Check.lt(100)],
        coerce=True
    ),
    'SEMAGESTAC':pa.Column(
        int, #dtype
        description = 'Quantidade de semanas da gestação.',           
        nullable=True , # nullable=False: Proíbe valores vazios (NaN)
        checks=[pa.Check.gt(4), pa.Check.lt(46)],
        coerce=True
    ),
    'QTDPARTNOR':pa.Column(
        int,
        description = 'Quantidade de partos normais antes do nascimento do bebê em questão.',
        nullable = False,
        checks = pa.Check.gt(-1),
        coerce = True
    ),
    'QTDPARTCES': pa.Column(
        int,
        description = 'Quantidade de partos cesarianos antes do nascimento do bebê em questão.',
        nullable = False,
        checks = pa.Check.gt(-1),
        coerce = True
    ),
    'CONSPRENAT': pa.Column(
        int,
        description = 'Quantidade de consultas prenatais feitas.',
        nullable = False,
        checks=[pa.Check.gt(-1), pa.Check.lt(100)],
        coerce = True
    ),
    'KOTELCHUCK': pa.Column(
        int,
        description = 'Índice de Adequação do Uso de Cuidados Pré-Natais (APNCU) - Avalia a qualidade do acompanhamento médico durante a gravidez.', #Quando o pré-natal começou e quantas consultas foram feitas em relação ao esperado para a idade gestacional.'
        nullable = False,
        checks=[pa.Check.gt(0), pa.Check.lt(10)],
        coerce = True        
    ),
    'DIFDATA': pa.Column(
        int,
        description = 'Diferença entre a data de registro da criança e data de nascimento.',
        nullable = False,
        checks = pa.Check.gt(-1),
        coerce = True
    ),
    'GRAVIDEZ': pa.Column(
        int,
        description = 'Indica se a gravidez foi simples, dupla, tripla e etc.',
        checks = pa.Check.isin([1,2,3,9]),
        coerce = True
    ),
    'PESO': pa.Column(
        float,
        description = 'Peso em gramas do recém-nascido.',
        checks=[pa.Check.gt(0), pa.Check.lt(7100.0)],
        coerce = True
    ),
    'HORANASC': pa.Column(
        "datetime64[ns]",
        description='Hora do nascimento do bebê',          
        nullable=True            
    ),
    'DTNASC': pa.Column(
        "datetime64[ns]", 
        description='Data de nascimento do bebê',
    ),
    'DATA_HORA_NASC': pa.Column(
        "datetime64[ns]",
        description='Data e hora combinadas do nascimento',
        nullable=True
    ),
    'SEXO': pa.Column(
        int,
        description = 'Sexo do recém-nascido',
        checks = pa.Check.isin([0,1,2]),
        nullable = False,
        coerce = True
    )
})

> Pyarrow

In [6]:
def limpeza_pyarrow(batch):
    colunas = [c for c in batch.schema.names if c not in ['IDADEPAI', 'IDADEMAE']]
    
    for col in colunas:
        batch = batch.filter(pc.is_valid(batch.column(col)))

    batch = batch.filter(pc.greater(batch.column('PESO'), 0))

    # Convertemos para string e removemos possíveis ".0" (comum se vier de float)
    dtnasc = batch.column('DTNASC').cast(py.string())
    dtnasc_limpo = pc.replace_substring(dtnasc, pattern=".0", replacement="")
    dtnasc_correto = pc.utf8_lpad(dtnasc_limpo, width=8, padding='0')
    
    hora = batch.column('HORANASC').cast(py.string())
    hora_limpo = pc.replace_substring(hora, pattern=".0", replacement="")
    hora_correta = pc.utf8_lpad(hora_limpo, width=4, padding='0')

    data_hora_str = pc.binary_join_element_wise(dtnasc_correto, hora_correta, " ")
    
    try:
        data_hora_full = pc.strptime(data_hora_str, format='%d%m%Y %H%M', unit='s')
        
    except py.ArrowInvalid:
        # Fallback caso algum dado esteja muito fora do padrão
        data_hora_full = py.array([None] * batch.num_rows, type = py.timestamp('s'))

    dtnasc_data = pc.strptime(dtnasc_correto, format='%d%m%Y', unit='s', error_is_null=True)
    hora_data = pc.strptime(hora_correta, format='%H%M', unit='s', error_is_null=True)
    data_hora_full = pc.strptime(data_hora_str, format='%d%m%Y %H%M', unit='s', error_is_null=True)

    idx_dtnasc = batch.schema.get_field_index('DTNASC')
    batch = batch.set_column(idx_dtnasc, 'DTNASC', dtnasc_data)

    idx_hrnasc = batch.schema.get_field_index('HORANASC')
    batch = batch.set_column(idx_hrnasc, 'HORANASC', hora_data)

    return batch.append_column('DATA_HORA_NASC', data_hora_full)

In [7]:
process = psutil.Process(os.getpid())

inicio = time.perf_counter()
memoria_inicio = process.memory_info().rss / 1024**2  # Em MB


arquivo = pq.read_table(f"{path}sinasc_reduzidissima_final.parquet")

print("Iniciando validação por lotes com PyArrow...")
print(f'{"*"*50}\n')

try:
    for batch in arquivo.to_batches(max_chunksize=n_amostra):
        batch_limpo = limpeza_pyarrow(batch)
        df_para_validar = batch_limpo.to_pandas() 
        schema.validate(df_para_validar)
        
        print(f"Lote de {batch_limpo.num_rows} linhas processado!")

except Exception as e:
    print(f"❌ Erro: {e}")

fim = time.perf_counter()
tempo = fim - inicio

memoria_fim = process.memory_info().rss / 1024**2

tabela_de_comparacao = [
    {
        "Biblioteca" : "Pyarrow",
        "Ação" : "Leitura e Validação do Schema",
        "Tempo" : tempo,
        "Memória Utilizada": memoria_fim - memoria_inicio
    }
]

print(f'{"*"*50}\n')
if(tempo<60):
    print(f'Tempo total de processamento: {tempo:.2f}s')
else:
    print(f'Tempo total de processamento: {tempo/60:.2f} min')
print(f'{"*"*50}\n')
print(f"Memória inicial: {memoria_inicio:.2f} MB")
print(f"Memória final: {memoria_fim:.2f} MB")
if(memoria_fim - memoria_inicio < 0):
    print(f"Memória total utilizada no processo: {(memoria_inicio - memoria_fim):.2f} MB")
else:
    print(f"Memória total utilizada no processo: {memoria_fim - memoria_inicio:.2f} MB")
print(f'{"*"*50}\n')

Iniciando validação por lotes com PyArrow...
**************************************************

Lote de 88675 linhas processado!
Lote de 95879 linhas processado!
Lote de 91110 linhas processado!
Lote de 91396 linhas processado!
Lote de 104920 linhas processado!
Lote de 92870 linhas processado!
Lote de 95696 linhas processado!
Lote de 98706 linhas processado!
Lote de 118963 linhas processado!
Lote de 98898 linhas processado!
Lote de 98243 linhas processado!
Lote de 99645 linhas processado!
Lote de 117901 linhas processado!
Lote de 96065 linhas processado!
Lote de 99360 linhas processado!
Lote de 110651 linhas processado!
Lote de 114246 linhas processado!
Lote de 108954 linhas processado!
Lote de 98762 linhas processado!
Lote de 123461 linhas processado!
Lote de 113199 linhas processado!
Lote de 103259 linhas processado!
Lote de 109076 linhas processado!
Lote de 128229 linhas processado!
Lote de 109679 linhas processado!
Lote de 108561 linhas processado!
Lote de 114339 linhas processado

> polars

In [8]:
def limpeza_polars(batch):
    colunas = [c for c in batch.columns if c not in ['IDADEPAI', 'IDADEMAE']]
    batch = batch.drop_nulls(subset = colunas)

    batch = batch.filter(pl.col("PESO")>0)
    batch = batch.with_columns(
        pl.col("DTNASC")
        .cast(pl.Utf8)
        .str.replace_all(r"\.0$", "")
        .str.strptime(pl.Date, format="%d%m%Y", strict=False)
    )
    
    batch = batch.with_columns(
        pl.col("HORANASC")
        .cast(pl.Utf8)
        .str.replace_all(r"\.0$", "")
        .str.strptime(pl.Datetime, format="%H%M", strict=False)
    )
    
    batch = batch.with_columns(
        pl.concat_str([pl.col("DTNASC"), pl.col("HORANASC")], separator=" ")
        .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M", strict=False)
        .alias("DATA_HORA_NASC")
    )
    
    batch = batch.drop_nulls(subset = colunas)

    return batch

In [9]:
process = psutil.Process(os.getpid())

inicio = time.perf_counter()
memoria_inicio = process.memory_info().rss / 1024**2  # Em MB

df_polars = pl.scan_parquet(f"{path}sinasc_reduzidissima_final.parquet")

print("Iniciando validação por lotes com Polars...")
print(f'{"*"*50}\n')

total_linhas = df_polars.select(pl.len()).collect().item()

for i in range(0, total_linhas, n_amostra):
    df_batch = df_polars.slice(i, n_amostra).collect()
    try:
        df_validar = limpeza_polars(df_batch)
        df_limpo_pandas = df_validar.to_pandas()
        schema.validate(df_limpo_pandas)
        
    except Exception as e:
        print(f"❌ Erro no bloco {i}: {e}")
    
    print(f"Lote de {len(df_limpo_pandas)} linhas processado!")

fim = time.perf_counter()
tempo = fim - inicio

memoria_fim = process.memory_info().rss / 1024**2

tabela_de_comparacao.append(
    {
        "Biblioteca" : "Polars",
        "Ação" : "Leitura e Validação do Schema",
        "Tempo" : tempo,
        "Memória Utilizada": memoria_fim - memoria_inicio
    }
)

print(f'{"*"*50}\n')
if(tempo<60):
    print(f'Tempo total de processamento: {tempo:.2f}s')
else:
    print(f'Tempo total de processamento: {tempo/60:.2f} min')
print(f'{"*"*50}\n')
print(f"Memória inicial: {memoria_inicio:.2f} MB")
print(f"Memória final: {memoria_fim:.2f} MB")
if(memoria_fim - memoria_inicio < 0):
    print(f"Memória total utilizada no processo: {(memoria_inicio - memoria_fim):.2f} MB")
else:
    print(f"Memória total utilizada no processo: {memoria_fim - memoria_inicio:.2f} MB")
print(f'{"*"*50}\n')

Iniciando validação por lotes com Polars...
**************************************************

Lote de 170579 linhas processado!
Lote de 180436 linhas processado!
Lote de 190432 linhas processado!
Lote de 193409 linhas processado!
Lote de 201624 linhas processado!
Lote de 208134 linhas processado!
Lote de 211510 linhas processado!
Lote de 221200 linhas processado!
Lote de 225573 linhas processado!
Lote de 228625 linhas processado!
**************************************************

Tempo total de processamento: 4.39s
**************************************************

Memória inicial: 1949.87 MB
Memória final: 2000.14 MB
Memória total utilizada no processo: 50.27 MB
**************************************************



> Dask

In [10]:
#pip install dask

In [11]:
import dask.dataframe as dd

In [12]:
def Limpeza_Dask(df_dask):
    colunas_para_limpar = df_dask.columns.difference(['IDADEPAI', 'IDADEMAE'])
    df_dask = df_dask.dropna(subset=colunas_para_limpar)
    df_dask = df_dask[df_dask['PESO'] > 0]
    
    df_dask['DTNASC'] = df_dask['DTNASC'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(8)
    df_dask['DTNASC'] = dd.to_datetime(df_dask['DTNASC'], format='%d%m%Y', errors='coerce')
    
    hr_str = df_dask['HORANASC'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(4)
    temp_hora = dd.to_datetime(hr_str, format='%H%M', errors='coerce')
    
    df_dask['DATA_HOUR_NASC'] = df_dask['DTNASC'] + \
                                dd.to_timedelta(temp_hora.dt.hour, unit='h') + \
                                dd.to_timedelta(temp_hora.dt.minute, unit='m')
    
    df_dask['HORANASC'] = temp_hora
    
    ddf_limpo = df_dask.dropna(subset=['DTNASC', 'DATA_HOUR_NASC'])
    
    return df_dask

In [13]:
process = psutil.Process(os.getpid())

inicio = time.perf_counter()
memoria_inicio = process.memory_info().rss / 1024**2  # Em MB

df_dask = dd.read_parquet(f"{path}sinasc_reduzidissima_final.parquet")

print("Iniciando validação do schema 'com Dask'...")
print(f'{"*"*50}\n')

ddf_limpo = Limpeza_Dask(df_dask)

try:
    schema.validate(ddf_limpo)    
except pa.errors.SchemaError as e:
    print(f"❌ Erro de validação encontrado: {e}")

fim = time.perf_counter()
tempo = fim - inicio

memoria_fim = process.memory_info().rss / 1024**2

tabela_de_comparacao.append(
    {
        "Biblioteca" : "Dask",
        "Ação" : "Leitura e Validação do Schema",
        "Tempo" : tempo,
        "Memória Utilizada": memoria_fim - memoria_inicio
    }
)

print(f'{"*"*50}\n')
if(tempo<60):
    print(f'Tempo total de processamento: {tempo:.2f}s')
else:
    print(f'Tempo total de processamento: {tempo/60:.2f} min')
print(f'{"*"*50}\n')
print(f"Memória inicial: {memoria_inicio:.2f} MB")
print(f"Memória final: {memoria_fim:.2f} MB")
if(memoria_fim - memoria_inicio < 0):
    print(f"Memória total utilizada no processo: {(memoria_inicio - memoria_fim):.2f} MB")
else:
    print(f"Memória total utilizada no processo: {memoria_fim - memoria_inicio:.2f} MB")
print(f'{"*"*50}\n')

Iniciando validação do schema 'com Dask'...
**************************************************

**************************************************

Tempo total de processamento: 0.09s
**************************************************

Memória inicial: 1949.11 MB
Memória final: 1899.47 MB
Memória total utilizada no processo: 49.64 MB
**************************************************



> Pandas

In [14]:
#pip install fastparquet

In [15]:
from fastparquet import ParquetFile

In [16]:
def Limpeza_pandas(df):
    colunas_para_limpar = df.columns.difference(['IDADEPAI', 'IDADEMAE'])
    df = df.dropna(subset=colunas_para_limpar)
    df = df[df['PESO'] > 0]
    
    df['DTNASC'] = pd.to_datetime(
        df['DTNASC'].astype(str).fillna('0').str.zfill(8), 
        format='%d%m%Y', 
        errors='coerce'
    )
    
    df['HORANASC'] = pd.to_datetime(
        df['HORANASC'].astype(str).fillna('0').str.zfill(4), 
        format='%H%M', 
        errors='coerce'
    ).dt.time
    
    datas_str = df['DTNASC'].astype(str)
    horas_str = df['HORANASC'].astype(str)

    # 3. Combina e converte de volta. O 'errors=coerce' vai transformar o que era 'NaT' em nulo novamente
    df['DATA_HORA_NASC'] = pd.to_datetime(
        datas_str + ' ' + horas_str, 
        errors='coerce'
    )
    
    df = df.dropna(subset=colunas_para_limpar)
    return df

In [17]:
process = psutil.Process(os.getpid())

inicio = time.perf_counter()
memoria_inicio = process.memory_info().rss / 1024**2  # Em MB

file_name = f"{path}sinasc_reduzidissima_final.parquet"

print("Iniciando validação com Pandas...")
print(f'{"*"*50}\n')

# Carrega os metadados do arquivo
pf = ParquetFile(file_name)
tamanho = pf.count()
print(f'{tamanho}')
# pf.iter_row_groups() lê o arquivo conforme ele foi particionado originalmente
# Para garantir exatamente 500k linhas, usamos pf.to_pandas com os índices de linha:
for chunk in pf.iter_row_groups():
    # Executa sua função de limpeza
    df_processado = Limpeza_pandas(chunk)
    try:
        schema.validate(df_processado)
    except pa.errors.SchemaError as e:
        print(f"❌ Erro de validação: {e}")
        
fim = time.perf_counter()
tempo = fim - inicio

memoria_fim = process.memory_info().rss / 1024**2

tabela_de_comparacao.append(
    {
        "Biblioteca" : "Pandas",
        "Ação" : "Leitura e Validação do Schema",
        "Tempo" : tempo,
        "Memória Utilizada": memoria_fim - memoria_inicio
    }
)

print(f'{"*"*50}\n')
if(tempo<60):
    print(f'Tempo total de processamento: {tempo:.2f}s')
else:
    print(f'Tempo total de processamento: {tempo/60:.2f} min')
print(f'{"*"*50}\n')
print(f"Memória inicial: {memoria_inicio:.2f} MB")
print(f"Memória final: {memoria_fim:.2f} MB")
if(memoria_fim - memoria_inicio < 0):
    print(f"Memória total utilizada no processo: {(memoria_inicio - memoria_fim):.2f} MB")
else:
    print(f"Memória total utilizada no processo: {memoria_fim - memoria_inicio:.2f} MB")
print(f'{"*"*50}\n')

Iniciando validação com Pandas...
**************************************************

5000000
**************************************************

Tempo total de processamento: 13.45s
**************************************************

Memória inicial: 1901.30 MB
Memória final: 1898.32 MB
Memória total utilizada no processo: 2.98 MB
**************************************************



---

# 2. Operações em Grandes Volumes de dados, incrementado gradativamente

In [18]:
import numpy as np
intervalos_iniciais = [
    pd.Timedelta(hours=12),   # 2.1 - 12 Horas
    pd.Timedelta(days=1),    # 2.2 - 1 Dia
]

intervalos_medianos = [
    pd.Timedelta(weeks=1),      # 2.3 - 1 Semana
    pd.offsets.MonthBegin(1),   # 2.4 - 1 Mês (Lógica de calendário)
]

intervalos_finais = [
    pd.offsets.QuarterBegin(1), # 2.5 - 1 Trimestre
    pd.offsets.SemiMonthEnd(12),# 2.6 - 1 Semestre (ou 6 * MonthBegin)
]

intervalos_alemDoFim = [
    pd.offsets.YearBegin(1),    # 2.7 - 1 Ano
    pd.offsets.YearBegin(10),   # 2.8 - Limite (Aprox. 522 semanas = 10 anos)
]

In [20]:
meta = df_batch.columns.tolist()
meta

AttributeError: 'list' object has no attribute 'tolist'

In [ ]:
data_base = df_batch['DATA_HORA_NASC'].min() ##Pega a data mais antiga.

In [ ]:
#pip install scipy

In [ ]:
from scipy import stats
from scipy.stats import probplot
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpaches
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COR_PRINCIPAL = '#2E75B6'
COR_DESTAQUE = '#C62828'
COR_MEDIA = '#E65100' #laranja
COR_MEDIANA = '#2E7D32' # verde
COR_IQR = '#6A1B9A' # roxo
COR_DESVIO = '#AD1457' # rosa

COR_OP = {
    'HV0002': '#607D8B', #juno -> azul-cinza
    'HV0003': '#1565C0', #uber -> azul escuro
    'HV0004': '#E65100', #via -> laranja
    'HV0005': '#AD1457', #lyft -> rosa
}

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})
import seaborn, plotly

In [ ]:
df_batch.head()

In [21]:
def grafico_de_sturges(df_incremento, tempo):
    print(f'{tempo}')
    df_colunas = df_incremento[df_incremento['IDADEMAE','IDADEPAI']]
    
    for i in df_colunas:
        serie = df_colunas[i]
        
        fig, ax   = plt.subplots(figsize =(6, 3))
        
        media   = serie.mean() # media
        mediana = serie.median() # mediana
        dp      = serie.std() # Mede a dispersão; quanto os dados variam em relação à média.
        q1      = serie.quantile(0.25) 
        q3      = serie.quantile(0.75)
        p99     = serie.quantile(0.99) # Valor que delimita os 1% maiores dados da série.
        skew    = serie.skew() # Mede a falta de simetria da distribuição.
    
        
        series_vis = serie[serie<= p99]
        n_fora    = (serie > p99).sum()
    
        
        #desvio padrão
        #avxspan - Define uma área de sombreamento
        # 1° linha: define o tamanho do sombreamento. (0, media-dp), onde começa e media + dp, one termina.
        # 2° linha: define a transparência da área.
        ax.axvspan(max(0, media - dp), media +dp,
                   alpha = 0.08, color=COR_DESVIO,
                   label = f'Media desv (desv = {dp:.2f})')
        ax.axvline(max(0, media - dp), color=COR_DESVIO, lw=1.2, ls=':') # define a linha de demarcação
        ax.axvline( (media + dp), color=COR_DESVIO, lw=1.2, ls=':') # define a linha de demarcação
    
        #IQR
        ax.axvspan(q1, q3, alpha = 0.12 ,color=COR_IQR,
                   label = f'IQR: Q1 = {q1:.2f} - q3 = {q3:.2f}')
        ax.axvline(q1, color=COR_IQR, lw=1.4, ls=':')
        ax.axvline(q3, color=COR_IQR, lw=1.4, ls=':')
        
        #Medidas de localização
        ax.axvline(media, color =  COR_MEDIA, lw = 2.4, ls="--", label=f'Média = {media:.2f}')
        ax.axvline(mediana, color =  COR_MEDIANA, lw = 2.4, ls="--", label=f'Mediana = {mediana:.2f}')
    
        
        ymax = ax.get_ylim()[1]
        
        #Mediana linha
        ax.annotate(f'Mediana\n {mediana:.2f}',
                   xy=(mediana, ymax *0.72),
                   xytext = (mediana - 18, ymax *0.92),
                   color = COR_MEDIANA, fontsize =10, fontweight ='bold',
                   ha = 'center',
                   arrowprops = dict(arrowstyle = '->', color = COR_MEDIANA, lw=1.8),
                   bbox = dict(boxstyle = 'round, pad=0.3', facecolor = 'white',
                                     edgecolor= COR_MEDIANA, alpha = 0.9),
        )
    
        #media linha
        ax.annotate(f'Media\n {media:.2f}',
                   xy=(media, ymax *0.60),
                   xytext = (media + 20, ymax *0.40),
                   color = COR_MEDIA, fontsize =10, fontweight ='bold',
                   ha = 'center',
                   arrowprops = dict(arrowstyle = '->', color = COR_MEDIA, lw=1.8),
                   bbox = dict(boxstyle = 'round, pad=0.3', facecolor = 'white',
                                     edgecolor= COR_MEDIA, alpha = 0.9),
        )
        
        # Fórmula de "sturges"
        n = len(meta)
        k = int(1 + 3.322 * np.log10(n))
        
        print(f"Número de classes (Sturges): {k}")
        
        # Histograma
        plt.hist(df_incremento[i], bins=k, color = 'salmon', edgecolor='red')
        plt.title(f"Histograma (Regra de Sturges)")
        plt.xlabel(f"{i}")
        plt.ylabel("Frequência")
        plt.show()
    
        
        distancia = (df_colunas[i].mean() - df_colunas[i].median())
        assimetria = ''
        if (distancia > 0):
            assimetria = 'Assimetria à direita - POSITIVA!'
        elif(distancia < 0):
            assimetria = 'Assimetria à esquerda - NEGATIVA!'
        else:
            assimetria = 'Simétrica'
        
        print(f"Distância Média/Mediana: {distancia:.2f}\n"
              f'{assimetria}\n'
              f'{'_'*120}\n'
             )

In [ ]:
def grafico_de_correlacao(df_incremento, tempo):
    print(f'{tempo}')
    df_colunas = df_incremento[df_incremento['IDADEMAE','IDADEPAI']]
    
    for i in meta:
        serie = df_incremento[i]
        
        fig, ax   = plt.subplots(figsize =(6, 3))
        
        media   = serie.mean() # media
        mediana = serie.median() # mediana
        dp      = serie.std() # Mede a dispersão; quanto os dados variam em relação à média.
        q1      = serie.quantile(0.25) 
        q3      = serie.quantile(0.75)
        p99     = serie.quantile(0.99) # Valor que delimita os 1% maiores dados da série.
        skew    = serie.skew() # Mede a falta de simetria da distribuição.
    
        
        series_vis = serie[serie<= p99]
        n_fora    = (serie > p99).sum()
    
        
        #desvio padrão
        #avxspan - Define uma área de sombreamento
        # 1° linha: define o tamanho do sombreamento. (0, media-dp), onde começa e media + dp, one termina.
        # 2° linha: define a transparência da área.
        ax.axvspan(max(0, media - dp), media +dp,
                   alpha = 0.08, color=COR_DESVIO,
                   label = f'Media desv (desv = {dp:.2f})')
        ax.axvline(max(0, media - dp), color=COR_DESVIO, lw=1.2, ls=':') # define a linha de demarcação
        ax.axvline( (media + dp), color=COR_DESVIO, lw=1.2, ls=':') # define a linha de demarcação
    
        #IQR
        ax.axvspan(q1, q3, alpha = 0.12 ,color=COR_IQR,
                   label = f'IQR: Q1 = {q1:.2f} - q3 = {q3:.2f}')
        ax.axvline(q1, color=COR_IQR, lw=1.4, ls=':')
        ax.axvline(q3, color=COR_IQR, lw=1.4, ls=':')
        
        #Medidas de localização
        ax.axvline(media, color =  COR_MEDIA, lw = 2.4, ls="--", label=f'Média = {media:.2f}')
        ax.axvline(mediana, color =  COR_MEDIANA, lw = 2.4, ls="--", label=f'Mediana = {mediana:.2f}')
    
        
        ymax = ax.get_ylim()[1]
        
        #Mediana linha
        ax.annotate(f'Mediana\n {mediana:.2f}',
                   xy=(mediana, ymax *0.72),
                   xytext = (mediana - 18, ymax *0.92),
                   color = COR_MEDIANA, fontsize =10, fontweight ='bold',
                   ha = 'center',
                   arrowprops = dict(arrowstyle = '->', color = COR_MEDIANA, lw=1.8),
                   bbox = dict(boxstyle = 'round, pad=0.3', facecolor = 'white',
                                     edgecolor= COR_MEDIANA, alpha = 0.9),
        )
    
        #media linha
        ax.annotate(f'Media\n {media:.2f}',
                   xy=(media, ymax *0.60),
                   xytext = (media + 20, ymax *0.40),
                   color = COR_MEDIA, fontsize =10, fontweight ='bold',
                   ha = 'center',
                   arrowprops = dict(arrowstyle = '->', color = COR_MEDIA, lw=1.8),
                   bbox = dict(boxstyle = 'round, pad=0.3', facecolor = 'white',
                                     edgecolor= COR_MEDIA, alpha = 0.9),
        )
        
        # Fórmula de "sturges"
        n = len(meta)
        k = int(1 + 3.322 * np.log10(n))
        
        print(f"Número de classes (Sturges): {k}")
        
        # Histograma
        plt.hist(df_incremento[i], bins=k, color = 'salmon', edgecolor='red')
        plt.title(f"Histograma (Regra de Sturges)")
        plt.xlabel(f"{i}")
        plt.ylabel("Frequência")
        plt.show()
    
        
        distancia = (df_incremento[i].mean() - df_incremento[i].median())
        assimetria = ''
        if (distancia > 0):
            assimetria = 'Assimetria à direita - POSITIVA!'
        elif(distancia < 0):
            assimetria = 'Assimetria à esquerda - NEGATIVA!'
        else:
            assimetria = 'Simétrica'
        
        print(f"Distância Média/Mediana: {distancia:.2f}\n"
              f'{assimetria}\n'
              f'{'_'*120}\n'
             )

## O que analisar
* Quantos bebês nasceram, crescendo gradativamente
* Gráfico de Sturges sobre a idade das mães
* Tabela de correlação para colunas de Consultas Prenatais, Kotelchuck e Semanas de Gestação

In [ ]:
data_inicio = df_batch['DTNASC'].min()
mask = (df_batch['DTNASC'] >= data_inicio) & (df_batch['DTNASC'] < data_inicio + intervalos_medianos[1])
df_primeiro_mes = df_batch[mask]

In [ ]:
for tempo in intervalos_iniciais:
    data_limite = data_base + tempo
    df_incremento = df_batch[df_batch['DATA_HORA_NASC'] <= data_limite]
    
    grafico_de_sturges(df_incremento, tempo)

In [ ]:
for tempo in intervalos_medianos:
    data_limite = data_base + tempo
    df_incremento = df_batch[df_batch['DATA_HORA_NASC'] <= data_limite]
    
    grafico_de_sturges(df_incremento, tempo)

In [ ]:
for tempo in intervalos_finais:
    data_limite = data_base + tempo
    df_incremento = df_batch[df_batch['DATA_HORA_NASC'] <= data_limite]
    
    grafico_de_sturges(df_incremento, tempo)

In [ ]:
for tempo in intervalos_alemDoFim:
    data_limite = data_base + tempo
    df_incremento = df_batch[df_batch['DATA_HORA_NASC'] <= data_limite]
    
    grafico_de_sturges(df_incremento, tempo)

---

# 3. Comparação de Desempenho, envolvendo as bibliotecas utilizadas em sala

---
# Referências
**Arrumar essas referências**
> 
> Link do Dataframe: https://dadosabertos.saude.gov.br/dataset/sistema-de-informacao-sobre-nascidos-vivos-sinasc
>
> Links de retiradas de dúvidas com Gemini:
> * https://gemini.google.com/share/ca16ce21a4ad
> * https://gemini.google.com/share/03d3bfa5343c
> * https://gemini.google.com/share/406b9e8a584f